# 012 Classification Ladder Error Review

This notebook reviews dev-only classification ladder errors. It does not train, embed, call an LLM, generate answers, or score holdout.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LADDER = ROOT / 'data' / 'eval' / 'classification_ladder_dev'
OUT = ROOT / 'data' / 'eval' / 'classification_ladder_error_review_dev'
SUMMARY_MD = OUT / 'classification_error_review_summary_dev.md'

assert (LADDER / 'classification_ladder_summary_dev.json').exists(), 'Run notebook 011 first.'

## Build Artifacts

In [ ]:
RUN_REVIEW = True

if RUN_REVIEW:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'analyze_classification_ladder_errors.py'),
        ],
        cwd=ROOT,
        check=True,
    )

assert SUMMARY_MD.exists(), f'Missing error review summary: {SUMMARY_MD}'

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))

## Review Tables

In [ ]:
primary_errors = pd.read_csv(OUT / 'classification_primary_error_counts_dev.csv')
primary_examples = pd.read_csv(OUT / 'classification_primary_error_examples_dev.csv')
multilabel_gaps = pd.read_csv(OUT / 'classification_multilabel_gap_counts_dev.csv')
multilabel_examples = pd.read_csv(OUT / 'classification_multilabel_gap_examples_dev.csv')
rank_buckets = pd.read_csv(OUT / 'classification_ranked_primary_buckets_dev.csv')
safety_gaps = pd.read_csv(OUT / 'classification_safety_signal_gap_counts_dev.csv')
safety_examples = pd.read_csv(OUT / 'classification_safety_signal_gap_examples_dev.csv')

display(primary_errors.head(15))
display(multilabel_gaps.head(15))
display(safety_gaps.head(15))

## Primary-Domain Errors

In [ ]:
primary_totals = primary_errors.groupby('profile', as_index=False)['records'].sum()
display(primary_totals)

ax = primary_totals.set_index('profile')['records'].plot(kind='bar', figsize=(6, 3.5), rot=0)
ax.set_title('Primary-domain errors by profile')
ax.set_xlabel('Profile')
ax.set_ylabel('Errors')
plt.tight_layout()

top_primary = primary_errors.head(12).copy()
top_primary['pair'] = top_primary['expected_primary_domain'] + ' -> ' + top_primary['predicted_primary_domain']
ax = top_primary.set_index('pair')['records'].plot(kind='barh', figsize=(9, 5))
ax.set_title('Top primary-domain confusions')
ax.set_xlabel('Records')
ax.set_ylabel('Expected -> predicted')
plt.tight_layout()

## Multi-Label Gaps

In [ ]:
missing = multilabel_gaps[multilabel_gaps['gap_type'] == 'missing_expected']
missing_plot = missing.pivot(index='domain', columns='profile', values='records').fillna(0)
ax = missing_plot.plot(kind='barh', figsize=(9, 5))
ax.set_title('Missing expected domain labels')
ax.set_xlabel('Records')
ax.set_ylabel('Domain')
plt.tight_layout()

display(multilabel_gaps.sort_values('records', ascending=False).head(20))

## Ranked Primary Buckets

In [ ]:
bucket_totals = rank_buckets.groupby(['profile', 'rank_bucket'], as_index=False)['records'].sum()
bucket_order = ['rank_1', 'rank_2', 'rank_3', 'rank_gt_3', 'not_predicted']
bucket_pivot = bucket_totals.pivot(index='rank_bucket', columns='profile', values='records').reindex(bucket_order).fillna(0)
display(bucket_pivot)

ax = bucket_pivot.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_title('Expected primary-domain rank buckets')
ax.set_xlabel('Rank bucket')
ax.set_ylabel('Records')
plt.tight_layout()

## Safety Signal Gaps

In [ ]:
missed_safety = safety_gaps[safety_gaps['gap_type'] == 'missed_expected']
safety_plot = missed_safety.pivot(index='safety_signal', columns='profile', values='records').fillna(0)
ax = safety_plot.plot(kind='barh', figsize=(9, 4))
ax.set_title('Missed expected safety signals')
ax.set_xlabel('Records')
ax.set_ylabel('Safety signal')
plt.tight_layout()

display(safety_gaps)

## Example Queues

In [ ]:
pd.set_option('display.max_colwidth', 120)
display(primary_examples.head(20))
display(multilabel_examples.head(20))
display(safety_examples.head(20))

## Working Decision

The transparent baseline's biggest classification gap is not primary-domain ranking alone. It is missing dense secondary expected labels, especially OS/kernel/driver, application software, hardware, and identity/access labels. The next lightweight baseline should therefore test whether source tags or nearest-neighbor examples can improve secondary-label recall without destroying primary-domain precision.